# Open-XTTS-Trainer: End-to-End 3-Stage Training Demo

This notebook demonstrates how to train or fine-tune all 3 stages of **XTTS v2**:
1. **Stage 2:** Text Tokenizer & Vocabulary Extension for New Scripts (e.g. Arabic, Hindi, Cyrillic) + Autoregressive GPT Training
2. **Stage 3:** Latent HiFi-GAN Vocoder (Extracting temporal GPT latents & fine-tuning generator + discriminator)
3. **Stage 1:** Discrete VAE (DVAE) Audio Tokenizer fine-tuning at 22,050 Hz

## 1. Setup & Environment

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Check GPU availability
!nvidia-smi

# Install open-xtts-trainer and Idiap's Coqui TTS fork
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!pip install -q torchcodec
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .

## 2. Download Pretrained XTTS-v2 Base Checkpoints & Setup Speech Data

In [ ]:
# 1. Download official XTTS v2 base weights (vocab, dvae, config, model)
!mkdir -p xtts_base
!wget -q -O xtts_base/vocab.json https://huggingface.co/coqui/XTTS-v2/raw/main/vocab.json
!wget -q -O xtts_base/config.json https://huggingface.co/coqui/XTTS-v2/raw/main/config.json
!wget -q -O xtts_base/dvae.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth
!wget -q -O xtts_base/model.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth

# 2. Download 30.0 Minutes of Clean Human Speech (600 training clips)
!python -m xtts_trainer.speech_data --minutes 30 --output_dir sample_audio


## 3. Stage 2: GPT Alignment & Vocabulary Extension


In [ ]:
# 1. Sample target language text (e.g. Arabic, Hindi, Dhivehi)
with open("sample_target.txt", "w", encoding="utf-8") as f:
    f.write("ދިވެހި ބަސް. สวัสดี. مرحبا. नमस्ते")

# 2. Extend Tokenizer for new alphabet (resizes text_embedding and text_head)
!python xtts_trainer/stage2_gpt/vocab_extender.py --base_vocab xtts_base/vocab.json --target_text sample_target.txt --output_vocab xtts_base/vocab_extended.json

# 3. Fine-Tune Autoregressive GPT Backbone with true DVAE acoustic target cross-entropy
!python xtts_trainer/stage2_gpt/train.py --audio_dir sample_audio --vocab_path xtts_base/vocab_extended.json --checkpoint_dir xtts_base --output_dir checkpoints/gpt --epochs 5 --batch_size 4 --device cuda


## 4. Stage 3: Latent HiFi-GAN Vocoder Fine-Tuning


In [ ]:
# 1. Extract true per-frame temporal GPT latents & speaker embeddings (~2 mins on T4 GPU)
!python xtts_trainer/stage3_hifigan/extract_latents.py --audio_dir sample_audio --output_dir extracted_latents --checkpoint_dir xtts_base --device cuda

# 2. Fine-tune Latent HiFi-GAN Vocoder starting from pretrained base weights (25 Epochs on 600 clips)
!python xtts_trainer/stage3_hifigan/train.py --audio_dir sample_audio --latents_dir extracted_latents --checkpoint_dir xtts_base --output_dir checkpoints/hifigan --batch_size 16 --epochs 25 --device cuda


## 5. Stage 1: Discrete VAE (DVAE) Fine-Tuning

In [ ]:
# Run DVAE fine-tuning at native 22,050 Hz on the 600 speech clips
!python xtts_trainer/stage1_dvae/train.py --audio_dir sample_audio --output_dir checkpoints/dvae --continue_path xtts_base/dvae.pth --batch_size 16 --epochs 25 --device cuda


## 6. View Checkpoints & Test Synthesis with Trained Weights

In [ ]:
# 1. List all saved checkpoints across all stages
print("=== Saved Checkpoints on Disk ===")
!find checkpoints/ -name "*.pt" -o -name "*.pth" 2>/dev/null

# 2. Synthesize using the fine-tuned Stage 2 GPT checkpoint!
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import IPython.display as ipd
import soundfile as sf

config = XttsConfig()
if os.path.exists("xtts_base/config.json"):
    config.load_json("xtts_base/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path="checkpoints/gpt/model.pth", vocab_path="checkpoints/gpt/vocab.json", eval=True)
model.cuda()

print("Synthesizing voice clone with trained model...")
outputs = model.synthesize(
    "Artificial intelligence is rapidly transforming how we communicate and build open technology every day.",
    config,
    speaker_wav="sample_audio/speech_001.wav",
    gpt_cond_len=6,
    language="en",
)
sf.write("cloned_speech.wav", outputs["wav"], 24000)
display(ipd.Audio("cloned_speech.wav"))
